# 01 — Data Exploration

Initial look at the reshaped Telco tables (`customers`, `products`, `interactions`).

Run `python scripts/run_pipeline.py` from the project root first to generate these files in `data/processed/`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"

customers = pd.read_csv(PROCESSED_DIR / "customers.csv", dtype={"customer_id": str})
products = pd.read_csv(PROCESSED_DIR / "products.csv")
interactions = pd.read_csv(PROCESSED_DIR / "interactions.csv", dtype={"customer_id": str})

print(customers.shape, products.shape, interactions.shape)

## Customers overview

In [ ]:
customers.head()

In [ ]:
customers.describe(include="all")

## Product popularity

Which products/services have the most subscribers? This is a useful sanity check
and a naive "most popular" baseline to compare smarter models against later.

In [ ]:
product_counts = (
    interactions.merge(products, on="product_id")
    .groupby("product_name")
    .size()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
product_counts.plot(kind="barh")
plt.xlabel("Number of subscribed customers")
plt.title("Product popularity")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Products per customer

Distribution of how many products each customer currently subscribes to —
relevant for understanding cross-sell headroom.

In [ ]:
products_per_customer = interactions.groupby("customer_id").size()

plt.figure(figsize=(7, 4))
sns.histplot(products_per_customer, bins=range(0, products_per_customer.max() + 2), discrete=True)
plt.xlabel("Number of products subscribed")
plt.ylabel("Number of customers")
plt.title("Products per customer")
plt.tight_layout()
plt.show()

## Next steps

- Try `src.models.content_based.ContentBasedRecommender` on a few sample customers
- Try `src.models.collaborative_filtering.CollaborativeFilteringRecommender`
- Compare both against `src.models.hybrid.HybridRecommender`
- Evaluate with `src.evaluation.metrics.evaluate_all`